# 02 — Data Elaboration and Capacity
This notebook asks: *how are retained measurements organized into cycles, half-cycles, and steps, and what Capacity follows from them?* It runs independently on the tiny synthetic source and normally finishes in seconds. Canonical units are seconds, amperes, volts, ampere-hours, and watt-hours; positive current denotes charge. The example is instructional synthetic data, not a laboratory validation dataset.


In [ ]:
import tempfile
from pathlib import Path


def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples" / "data" / "synthetic_cycling.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the gregorithm repository folder.")


PROJECT_ROOT = find_project_root()
LOCAL_FILE = None  # Optional: Path(r"D:/your-local/canonical-file.csv")
SOURCE = (
    Path(LOCAL_FILE)
    if LOCAL_FILE is not None
    else PROJECT_ROOT / "examples" / "data" / "synthetic_cycling.csv"
)
_session = tempfile.TemporaryDirectory(prefix="gregorithm-notebook-")
WORK = Path(_session.name)
print(f"Input: {SOURCE.name}")
print(f"Temporary result area: {WORK}")

In [ ]:
from gregorithm import ElaborationRequest, ImportRequest, elaborate, import_source

imported = import_source(
    ImportRequest(SOURCE, WORK / "import", "synthetic-cell", "import-job", "import-v1")
)
elaborated = elaborate(
    ElaborationRequest(
        imported.ref.path / "points" / "canonical.parquet",
        imported.report.content_hash,
        WORK / "elaboration",
        "synthetic-cell",
        "elaboration-job",
        "elaboration-v1",
    )
)
print(f"Retained rows: {imported.report.retained_rows}")
print(f"Cycles: {elaborated.cycles}; half-cycles: {elaborated.half_cycles}")

In [ ]:
import pyarrow.parquet as pq

points = pq.read_table(elaborated.ref.path / "points" / "elaborated.parquet")
columns = ["time_s", "current_a", "voltage_v", "cycle", "half_cycle", "step"]
points.select(columns).to_pylist()

cycle and half_cycle are scientific labels derived from the documented current-sign rules. step codes are 0 = rest, 1 = constant current, 2 = constant voltage, and 3 = other. Rest classification does not erase measured current or energy.


In [ ]:
import matplotlib.pyplot as plt

time = points["time_s"].to_numpy()
current = points["current_a"].to_numpy()
voltage = points["voltage_v"].to_numpy()
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
axes[0].plot(time, current, marker="o", label="Current")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("Current (A)")
axes[0].legend()
axes[1].plot(time, voltage, marker="o", color="tab:orange", label="Voltage")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Voltage (V)")
axes[1].legend()
fig.suptitle("Synthetic cycling measurements")
fig.tight_layout()
fig

In [ ]:
from gregorithm import CapacityParameters, CapacityRequest, analyze_capacity

capacity_result = analyze_capacity(
    CapacityRequest(
        elaborated.ref.path,
        elaborated.fingerprint,
        WORK / "capacity",
        "synthetic-cell",
        "capacity-job",
        "capacity-v1",
        CapacityParameters(reference_cycle=1),
    )
)
capacity = pq.read_table(capacity_result.ref.path / "tables" / "capacity.parquet")
capacity.to_pylist()

In [ ]:
rows = capacity.to_pylist()
cycles = [row["cycle"] for row in rows]
charge = [
    float("nan") if row["charge_capacity_ah"] is None else row["charge_capacity_ah"] for row in rows
]
discharge = [
    float("nan") if row["discharge_capacity_ah"] is None else row["discharge_capacity_ah"]
    for row in rows
]
fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
ax.bar([c - width / 2 for c in cycles], charge, width, label="Charge")
ax.bar([c + width / 2 for c in cycles], discharge, width, label="Discharge")
ax.set_xlabel("Cycle")
ax.set_ylabel("Capacity (Ah)")
ax.set_title("Per-cycle Capacity")
ax.legend()
ax.set_xticks(cycles)
fig

Capacity and energy are integrated from retained real points using the documented trapezoidal rules. Efficiencies are percentages when both phases exist; None means unavailable, not zero. Mass- and area-normalized outputs require positive user-supplied physical metadata and are intentionally not invented here.
